<a href="https://colab.research.google.com/github/IGol22/SFML/blob/main/%D0%9A%D1%83%D1%80%D1%81%D0%BE%D0%B2%D0%B0%D1%8F/6_Classification_SI_median.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q catboost xgboost

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier, HistGradientBoostingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Подгрузка датасета с GitHub
file_name = "dataset_classic_ML.xlsx"
if not os.path.exists(file_name):
    raw_url = "https://raw.githubusercontent.com/IGol22/SFML/main/Курсовая/dataset_classic_ML.xlsx"
    !wget -q -O {file_name} "{raw_url}"

df = pd.read_excel(file_name)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00


,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,384.652,...,0,0,0,0,0,0,0,0,3,0
1,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,388.684,...,0,0,0,0,0,0,0,0,3,0
2,223.808778,161.142320,0.720000,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,446.808,...,0,0,0,0,0,0,0,0,3,0
3,1.705624,107.855654,63.235294,5.097360,5.097360,0.390603,0.390603,0.377846,41.862069,398.679,...,0,0,0,0,0,0,0,0,4,0
4,107.131532,139.270991,1.300000,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,466.713,...,0,0,0,0,0,0,0,0,0,0


In [2]:
# 1. Формируем таргет SI и исключаем утечки
target_col = 'SI'
targets_to_drop = ['IC50, mM', 'CC50, mM', 'SI']

X = df.drop(columns=targets_to_drop, errors='ignore').copy()

# Деление по медиане SI: 1 если значение > медианы, 0 если <= медианы
median_val = df[target_col].median()
y = (df[target_col] > median_val).astype(int)

print(f"Медиана SI: {median_val:.4f}")
print(f"Распределение классов:\n{y.value_counts()}")

# 2. Feature Engineering
if 'MolLogP' in X.columns and 'MolWt' in X.columns:
    X['MolLogP_x_MolWt'] = X['MolLogP'] * X['MolWt']

poly_cols = [c for c in ['MolLogP', 'MolWt'] if c in X.columns]
if poly_cols:
    poly = PolynomialFeatures(degree=2, include_bias=False)
    poly_feats = poly.fit_transform(X[poly_cols])
    poly_df = pd.DataFrame(poly_feats, columns=poly.get_feature_names_out(poly_cols), index=X.index)
    for col in poly_df.columns:
        if col not in X.columns:
            X[col] = poly_df[col]

if 'MolLogP' in X.columns:
    X['MolLogP_gt_3'] = (X['MolLogP'] > 3).astype(int)

# 3. Заполнение пропусков
if X.isnull().values.any():
    imputer = SimpleImputer(strategy='median')
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print(f"Размерность матрицы X: {X.shape}, Длина y: {len(y)}")

Медиана SI: 3.8462
Распределение классов:
SI
0    501
1    500
Name: count, dtype: int64
Размерность матрицы X: (1001, 215), Длина y: 1001


In [3]:
# Разбиение с сохранением баланса классов (stratify)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Отбор признаков и масштабирование
vt = VarianceThreshold(threshold=0.01)
X_train_sel = vt.fit_transform(X_train)
X_test_sel = vt.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)

# Модели
models = {
    'KNN': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'HistGradientBoosting': HistGradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0),
    'Stacking': StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(random_state=42)),
            ('gb', GradientBoostingClassifier(random_state=42)),
            ('xgb', XGBClassifier(random_state=42, eval_metric='logloss'))
        ],
        final_estimator=LogisticRegression()
    )
}

# Обучение
results = []
for name, model in models.items():
    tr_x = X_train_scaled if name in ['KNN'] else X_train_sel
    te_x = X_test_scaled if name in ['KNN'] else X_test_sel

    model.fit(tr_x, y_train)
    y_pred = model.predict(te_x)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(te_x)[:, 1]
    else:
        y_prob = y_pred

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    results.append({'Model': name, 'Accuracy': acc, 'F1': f1, 'ROC-AUC': roc_auc})

res_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False).round(3)
res_df

,Model,Accuracy,F1,ROC-AUC
0,KNN,0.662,0.673,0.705
6,CatBoost,0.657,0.639,0.690
5,XGBoost,0.647,0.616,0.681
2,Gradient Boosting,0.627,0.615,0.673
4,AdaBoost,0.637,0.578,0.670
1,Random Forest,0.642,0.604,0.664
7,Stacking,0.622,0.596,0.661
3,HistGradientBoosting,0.642,0.625,0.660


## Выводы и рекомендации

### Сравнение моделей
* **Лидер по качеству:** Наилучшие результаты показал алгоритм **KNN** (ROC-AUC = 0.705, Accuracy = 0.662, F1 = 0.673). Второе место занял **CatBoost** (ROC-AUC = 0.690).
* **Общий результат:** Средний уровень ROC-AUC для всех моделей составил 0.66–0.70. Относительное снижение точности объясняется природой показателя $SI$, определяемого как отношение $CC_{50} / IC_{50}$.

---

### Рекомендации
* **Выбор модели:** В качестве опорного алгоритма можно использовать **KNN** или **CatBoost**.
* **Практическое применение:** Для более точной оценки селективности предпочтительнее классифицировать $IC_{50}$ и $CC_{50}$ независимыми моделями с последующим вычислением $SI$.